# Cas test pseudo 2D pour tester le schéma Quick-Fram et Amont en Multiphase

In [ ]:

from trustutils.jupyter import plot
from trustutils import run
import os, sys


In [ ]:

run.introduction("J. FRANCESCATTO","01/02/2026")

run.TRUST_parameters()


In [ ]:

force_recalculation = True

schema = { "VDF" : "vdf  dis option_vdf { all_options }"}
convection = { "Amont" : "amont", "Quick" : "quick" }
mesh = { "201x5" :{ "NX" : 201,  "NY" : 5}, "401x5" :{ "NX" : 401,  "NY" : 5} }

facsec  = 0.2
number_of_partitions = 1
tmax    = 1.0

build_dir = run.BUILD_DIRECTORY

if force_recalculation or not os.path.exists(f'{build_dir}'):
    print('Recalculation running..')
    run.reset()
    for g in mesh:
        for s in schema.keys() :
            for m in convection.keys() :
                name = f"{s}_{g}_{m}"
                print(name)
                substitutions_dict = {"nx" : mesh[g]["NX"] ,
                                        "ny" : mesh[g]["NY"] ,
                                        "tmax": str(tmax), 
                                        "facsec" : str(facsec) ,
                                        "schema" : schema[s] ,
                                        "convection" : convection[m]
                                        }

                tc = run.addCaseFromTemplate("jdd.data",targetDirectory=f"{name}",dic=substitutions_dict,nbProcs=number_of_partitions)

                if number_of_partitions > 1:
                    tc.partition()
    run.printCases()
    run.runCases()
    perf = run.tablePerf() # tableau des performances de calcul
    display(perf)

else:
    print('No recalculation, reusing old results!')


In [ ]:

time = 1.0

Graph=plot.Graph(title="Advection en 1D à t = %s" %time,nY=1,nX=1,size=[12,5])
for g in mesh:
    for s in schema.keys() :
        for m in convection.keys() :

            name = f"{s}_{g}_{m}"
            name_plot = name #f"{s}_{m}"

            par = ""
            if number_of_partitions > 1 : par = "PAR_"
                
            Graph.addPlot(0,title=None)
            Graph.addSegment(
                build_dir+f"/{name}/{par}jdd_ALPHA2.son",
                time=time,
                label=f"{name_plot.replace('_','/')}"
                ) 
            
Graph.addPlot(0)
Graph.addSegment(
    build_dir+f"/{name}/{par}jdd_ALPHA.son", 
    time=time,
    label="Analytique", 
    color="black")
Graph.legend(loc="best")
Graph.label("x [m]", "Fraction de phase [-]")
Graph.visu(xmin=0.0,xmax=2.0)
                    